# Fase 6 - Baseline no Debian Remoto com Docker\n
Objetivo: validar se os 5 servicos sobem em containers, ficam estaveis e expostos nas portas esperadas.

## Pre-req\n
- Docker e Docker Compose instalados no servidor Debian\n
- Repositorio atualizado\n
- Arquivo .env.remote criado a partir do exemplo

In [1]:
import subprocess


def run(cmd):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result.returncode


run('cp -n .env.remote.example .env.remote || true')


$ cp -n .env.remote.example .env.remote || true



0

In [2]:
run('docker compose -f docker-compose.remote.yml --env-file .env.remote build')


$ docker compose -f docker-compose.remote.yml --env-file .env.remote build

couldn't find env file: /mnt/repositorio/workdir/repostorios/monorepo-ai-llm/spec/execucao/.env.remote



1

In [ ]:
run('docker compose -f docker-compose.remote.yml --env-file .env.remote up -d')
run('docker compose -f docker-compose.remote.yml ps')

In [ ]:
health_checks = [
    'curl -fsS http://localhost:3000/health',
    'curl -fsS http://localhost:3001/health',
    'curl -fsS http://localhost:3002/health',
    'curl -fsS http://localhost:3003/health',
    'curl -fsS http://localhost:3004/health',
]

for cmd in health_checks:
    run(cmd)

## Evidencias esperadas\n
- Todos os containers em estado Up\n
- Health endpoint respondendo em todas as portas\n
- Sem reinicios em loop

## Correcao de Syntax para Scripts Shell em Notebook

### Problema comum
Quando comandos shell sao colocados como texto com `\\n` literal em celula Python, o parser gera erro de sintaxe.

### Padrao correto
1. Em celula Python, execute shell usando `subprocess.run(..., shell=True)`.
2. Para blocos maiores, use `textwrap.dedent` e rode com `bash -lc`.
3. Evite colar codigo com `\\n` escapado no corpo da celula.

### Exemplo recomendado
Use a celula abaixo como modelo para comandos shell multi-linha com saida e tratamento de erro.

In [ ]:
# O que faz: executa script shell multi-linha com validacao de retorno
# Onde executa: kernel Python do notebook no host remoto
import subprocess
import textwrap

script = textwrap.dedent("""
set -e
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

docker compose -f docker-compose.remote.yml --env-file .env.remote config >/tmp/compose.rendered.yml
docker compose -f docker-compose.remote.yml --env-file .env.remote ps
""")

result = subprocess.run(
    ["bash", "-lc", script],
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Falha no script shell (exit={result.returncode})")

print("Script shell executado com syntax correta.")